# LibreMediaConverter – Colab GPU Test\nFührt nach dem manuellen Start einen 60-Sekunden-4K-Test aus. Lädt die Quelle von Google Drive, konvertiert mit NVENC und lädt das Ergebnis direkt nach Drive. GitHub speichert keine Videodaten.\n**Hinweis:** Der erste Test lädt die 11,2-GB-Quelle lokal in Colab. Die GPU-Laufzeit startet nicht automatisch durch GitHub.\n

In [ ]:
# GPU-Test: 60 Sekunden Zoomania nach 4K H.264 konvertieren und direkt zu Google Drive hochladen.
# Die Colab-Laufzeit muss manuell gestartet werden. GitHub kann sie nicht einschalten.
!pip -q install google-api-python-client
from google.colab import auth
auth.authenticate_user()
import google.auth, os, subprocess, shutil
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
from google.auth.exceptions import RefreshError
from pathlib import Path
creds, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/drive'])
drive = build('drive','v3',credentials=creds,cache_discovery=False)
source_id='1_fD9iNP8uDkV81L1ne-03sTEa6ERj1Ol'
work=Path('/content/gpu_convert'); work.mkdir(exist_ok=True)
source=work/'input.mp4'; output=work/'Zoomania_4K_GPU_Test.mp4'
print('GPU:'); subprocess.run(['nvidia-smi'],check=True)
print('Quelldatei wird aus Drive geladen (ca. 11,2 GB)...')
request=drive.files().get_media(fileId=source_id)
with open(source,'wb') as f:
    downloader=MediaIoBaseDownload(f,request,chunksize=16*1024*1024)
    done=False
    while not done:
        status,done=downloader.next_chunk()
        if status: print(f'Download: {status.progress()*100:.1f}%',end='\r')
print('\nPrüfe NVENC...')
enc=subprocess.run(['ffmpeg','-hide_banner','-encoders'],capture_output=True,text=True,check=True).stdout
if 'h264_nvenc' not in enc: raise RuntimeError('FFmpeg hat keinen h264_nvenc Encoder. Keine CPU-Konvertierung als GPU-Test.')
cmd=['ffmpeg','-hide_banner','-y','-t','60','-i',str(source),'-map','0:v:0','-map','0:a:0?','-vf','scale=3840:2160:flags=lanczos,format=yuv420p','-c:v','h264_nvenc','-b:v','20M','-maxrate','20M','-bufsize','40M','-c:a','aac','-b:a','384k','-ac','6','-movflags','+faststart',str(output)]
print('Konvertierung startet...')
subprocess.run(cmd,check=True)
def folder(name,parent=None):
    q="name = '"+name.replace("'","\\'")+"' and mimeType = 'application/vnd.google-apps.folder' and trashed = false"
    if parent: q+=" and '"+parent+"' in parents"
    r=drive.files().list(q=q,fields='files(id,name)',page_size=100).execute()['files']
    if r: return r[0]['id']
    return drive.files().create(body={'name':name,'mimeType':'application/vnd.google-apps.folder',**({'parents':[parent]} if parent else {})},fields='id').execute()['id']
root=folder('Video Converter'); target=folder('Fertig',root)
print('Lade Ergebnis direkt von Colab nach Google Drive hoch...')
media=MediaFileUpload(str(output),mimetype='video/mp4',resumable=True,chunksize=8*1024*1024)
req=drive.files().create(body={'name':output.name,'parents':[target]},media_body=media,fields='id,name,webViewLink')
result=None
while result is None:
    status,result=req.next_chunk()
    if status: print(f'Upload: {status.progress()*100:.1f}%',end='\r')
print('\nFERTIG:',result)
source.unlink(missing_ok=True); output.unlink(missing_ok=True)
